# **Configurations**

In [2]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [5]:
import sys

from google.colab import drive
drive.mount('/content/drive')

%cd /content/drive/MyDrive/College/FLEX-Med/backend/federated_learning

# Dependency Installation
!pip install fastapi uvicorn pyngrok nest-asyncio flwr["simulation"] flwr_datasets datasets supabase -q

PROJECT_ROOT = "/content/drive/MyDrive/College/FLEX-Med/backend/federated_learning"
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
/content/drive/MyDrive/College/FLEX-Med/backend/federated_learning


# **Verify if the environment is setup for FL simulation**

In [6]:
#!/usr/bin/env python3
"""
FL Health Verification Script v3.0 (Jupyter / Colab Safe)

Updated for v3.0 architecture:
- Checks 3-folder dataset structure (anchor, test, local_train)
- Validates Dirichlet partitioning configuration
- Verifies num-supernodes matches client count in data.json
- No longer checks deprecated per-client dataset folders
"""

import os
import json
import re
import sys
from pathlib import Path
from typing import Dict, List, Tuple

# =====================================================
# PROJECT ROOT RESOLUTION (JUPYTER SAFE)
# =====================================================
PROJECT_ROOT = Path.cwd()

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

# =====================================================
# ANSI COLORS
# =====================================================
class Colors:
    GREEN = '\033[92m'
    YELLOW = '\033[93m'
    RED = '\033[91m'
    BLUE = '\033[94m'
    BOLD = '\033[1m'
    END = '\033[0m'


def print_header(text: str):
    print(f"\n{Colors.BOLD}{Colors.BLUE}{'=' * 70}{Colors.END}")
    print(f"{Colors.BOLD}{Colors.BLUE}{text.center(70)}{Colors.END}")
    print(f"{Colors.BOLD}{Colors.BLUE}{'=' * 70}{Colors.END}\n")


def print_success(text: str):
    print(f"{Colors.GREEN}  {text}{Colors.END}")


def print_warning(text: str):
    print(f"{Colors.YELLOW}  {text}{Colors.END}")


def print_error(text: str):
    print(f"{Colors.RED}  {text}{Colors.END}")


def print_info(text: str):
    print(f"  {text}")


# =====================================================
# DATASET CHECK (v3.0: 3-folder structure)
# =====================================================
def check_dataset_structure(
    dataset_path: Path,
    expected_classes: List[str] = ["all", "hem"]
) -> Tuple[bool, Dict]:
    stats = {}

    if not dataset_path.exists():
        return False, {"error": f"Path does not exist: {dataset_path}"}

    for class_name in expected_classes:
        class_path = dataset_path / class_name
        if not class_path.exists():
            return False, {"error": f"Missing class directory: {class_name}"}

        image_exts = {'.jpg', '.jpeg', '.png', '.bmp'}
        images = [f for f in class_path.iterdir() if f.suffix.lower() in image_exts]
        stats[class_name] = len(images)

    total = sum(stats.values())
    if total == 0:
        return False, {"error": "No images found in dataset"}

    for cls in expected_classes:
        stats[f"{cls}_pct"] = stats[cls] / total * 100

    stats["total"] = total
    return True, stats


def verify_datasets(use_colab: bool = True) -> bool:
    print_header("Step 1: Dataset Structure (v3.0 - Dirichlet Partitioning)")

    if use_colab:
        base_path = Path("/content/drive/MyDrive/College/FLEX-Med/backend/datasets/cnmc")
    else:
        base_path = PROJECT_ROOT.parent / "datasets" / "cnmc"

    # v3.0 requires exactly 3 folders (no per-client datasets)
    datasets = {
        "Public Anchor (20%)": base_path / "cnmc_public_anchor",
        "Public Test (10%)":   base_path / "cnmc_public_test",
        "Local Train (70%)":   base_path / "cnmc_local_train",
    }

    all_valid = True

    for name, path in datasets.items():
        valid, stats = check_dataset_structure(path)
        if valid:
            print_success(f"{name}: {path}")
            print_info(f"  Total: {stats['total']} images")
            print_info(f"  Leukemia (all): {stats['all']:,} ({stats['all_pct']:.1f}%)")
            print_info(f"  Healthy  (hem): {stats['hem']:,} ({stats['hem_pct']:.1f}%)")
        else:
            print_error(f"{name}: MISSING")
            print_info(f"  {stats.get('error')}")
            all_valid = False

    if all_valid:
        print_info("")
        print_success("All 3 dataset folders present (v3.0 Dirichlet partitioning ready)")
    else:
        print_info("")
        print_error("Missing dataset folders. Required structure:")
        print_info(f"  {base_path}/")
        print_info(f"    cnmc_public_anchor/  (20% balanced anchor data)")
        print_info(f"    cnmc_public_test/    (10% balanced test data)")
        print_info(f"    cnmc_local_train/    (70% shared training data)")

    return all_valid


# =====================================================
# CLIENT CONFIG CHECK
# =====================================================
def verify_client_config() -> bool:
    print_header("Step 2: Client Configuration (client_data.json)")

    config_path = PROJECT_ROOT / "flex_med" / "utils" / "client_data.json"

    if not config_path.exists():
        print_warning(f"Config file not found: {config_path}")
        print_info("  This is normal before the first FL run.")
        print_info("  The file is auto-generated when /start_fl is called.")
        return True  # Not a failure, just not yet created

    if config_path.stat().st_size == 0:
        print_error(f"Config file is empty (0 bytes): {config_path}")
        print_info("  Action: Re-trigger FL from the frontend.")
        return False

    try:
        with open(config_path, 'r') as f:
            data = json.load(f)
    except json.JSONDecodeError as e:
        print_error(f"Corrupted JSON in {config_path}")
        print_info(f"  Error: {e}")
        return False

    # Handle both wrapped {"clients": [...]} and flat [...] formats
    clients = data.get('clients', data) if isinstance(data, dict) else data

    if not clients:
        print_error("No clients found in config.")
        return False

    sim_id = data.get('simulation_id', 'N/A') if isinstance(data, dict) else 'N/A'
    print_success(f"Loaded config: {config_path}")
    print_info(f"  Simulation ID: {sim_id}")
    print_info(f"  Number of clients: {len(clients)}")

    valid_models = ['resnet50', 'mobilenet_v2', 'densenet121', 'efficientnet_b0']

    for client in clients:
        c_id = client.get('id', '?')
        c_name = client.get('client_name', 'Unknown')
        c_model = client.get('model_type', 'Unknown')

        print_info(f"\n  Client {c_id}: {c_name}")
        print_info(f"    Model: {c_model}")

        if c_model not in valid_models:
            print_error(f"    Invalid model type! Must be one of: {valid_models}")
            return False

        model_path = Path(client.get('model_path', ''))
        if model_path.exists():
            print_warning(f"    Existing model: {model_path}")
        else:
            print_success(f"    Fresh training (no existing model)")

    return True


# =====================================================
# FL CONFIG & SUPERNODES CONSISTENCY CHECK
# =====================================================
def verify_fl_config() -> bool:
    print_header("Step 3: FL Configuration (pyproject.toml)")

    config_path = PROJECT_ROOT / "pyproject.toml"

    if not config_path.exists():
        print_error("pyproject.toml not found")
        return False

    content = config_path.read_text()
    params = {}

    for line in content.splitlines():
        line = line.strip()
        if '=' in line and not line.startswith('#') and not line.startswith('['):
            k, v = line.split('=', 1)
            params[k.strip()] = v.split('#')[0].strip().strip('"')

    print_success("FL config loaded")
    print_info(f"  Rounds:       {params.get('num-server-rounds', '?')}")
    print_info(f"  Local epochs: {params.get('local-epochs', '?')}")
    print_info(f"  Batch size:   {params.get('batch-size', '?')}")
    print_info(f"  LR:           {params.get('lr', '?')}")
    print_info(f"  LR decay:     {params.get('lr-decay', '?')}")
    print_info(f"  Distill LR:   {params.get('distill-lr', '?')}")
    print_info(f"  Distill epochs: {params.get('distill-epochs', '?')}")
    print_info(f"  Temperature:  {params.get('temperature', '?')}")

    # Check num-supernodes consistency with client_data.json
    supernodes_matches = re.findall(r'num-supernodes\s*=\s*(\d+)', content)
    if supernodes_matches:
        supernodes_values = list(set(supernodes_matches))
        num_supernodes = int(supernodes_values[0])

        if len(supernodes_values) > 1:
            print_error(f"    Inconsistent num-supernodes values: {supernodes_matches}")
            print_info(f"    Both entries in pyproject.toml must match!")
            return False

        print_info(f"  num-supernodes: {num_supernodes}")

        # Cross-check with client_data.json
        client_config_path = PROJECT_ROOT / "flex_med" / "utils" / "client_data.json"
        if client_config_path.exists() and client_config_path.stat().st_size > 0:
            try:
                with open(client_config_path) as f:
                    data = json.load(f)
                clients = data.get('clients', data) if isinstance(data, dict) else data
                num_clients = len(clients)

                if num_supernodes == num_clients:
                    print_success(f"    num-supernodes ({num_supernodes}) matches client count ({num_clients})")
                else:
                    print_error(f"    MISMATCH: num-supernodes={num_supernodes} but client_data.json has {num_clients} clients!")
                    print_info(f"    This WILL cause 'Partition ID exceeds number of clients' errors.")
                    print_info(f"    Fix: Update pyproject.toml or re-trigger /start_fl.")
                    return False
            except (json.JSONDecodeError, Exception):
                pass
    else:
        print_warning("  num-supernodes not found in pyproject.toml")

    return True


# =====================================================
# DIRICHLET CONFIG CHECK
# =====================================================
def verify_dirichlet_config() -> bool:
    print_header("Step 4: Dirichlet Partitioning Configuration")

    alpha = os.getenv("FLEX_MED_DIRICHLET_ALPHA", "not set (default: 0.5)")
    seed = os.getenv("FLEX_MED_DIRICHLET_SEED", "not set (default: 42)")
    min_size = os.getenv("FLEX_MED_DIRICHLET_MIN_PARTITION_SIZE", "not set (default: 100)")
    train_path = os.getenv("LOCAL_TRAIN_DATASET_PATH", "not set (uses default)")

    print_info(f"  FLEX_MED_DIRICHLET_ALPHA:              {alpha}")
    print_info(f"  FLEX_MED_DIRICHLET_SEED:               {seed}")
    print_info(f"  FLEX_MED_DIRICHLET_MIN_PARTITION_SIZE:  {min_size}")
    print_info(f"  LOCAL_TRAIN_DATASET_PATH:               {train_path}")

    print_info("")
    print_info("  Alpha guide:")
    print_info("    0.1  = Extreme heterogeneity (very skewed)")
    print_info("    0.5  = Moderate heterogeneity (default)")
    print_info("    1.0  = Mild heterogeneity")
    print_info("    10.0 = Nearly IID (uniform)")

    print_success("Dirichlet config OK (runtime partitioning)")
    return True


# =====================================================
# CLEAN STATE CHECK
# =====================================================
def verify_clean_state(use_colab: bool = True) -> bool:
    print_header("Step 5: Model Checkpoints & Clean State")

    models_dir = (
        Path("/content/drive/MyDrive/College/FLEX-Med/backend/models")
        if use_colab
        else PROJECT_ROOT.parent / "models"
    )

    if not models_dir.exists():
        print_success("No models directory (fresh start)")
        return True

    checkpoints = list(models_dir.glob("*.pt"))
    if checkpoints:
        print_warning(f"Found {len(checkpoints)} existing checkpoint(s):")
        for cp in checkpoints:
            size_mb = cp.stat().st_size / (1024 * 1024)
            print_info(f"    {cp.name} ({size_mb:.1f} MB)")
        print_info("  These will be resumed from if model_path matches.")
    else:
        print_success("No checkpoints found (clean start)")


    return True


# =====================================================
# MAIN
# =====================================================
def main():
    print(f"\n{Colors.BOLD}FLEX-Med FL Health Verification (v3.0){Colors.END}")
    print(f"{Colors.BOLD}Dirichlet Partitioning | Runtime Data Heterogeneity{Colors.END}")

    results = {
        "Datasets":    verify_datasets(use_colab=True),
        "Client Config": verify_client_config(),
        "FL Config":   verify_fl_config(),
        "Dirichlet":   verify_dirichlet_config(),
        "Clean State": verify_clean_state(use_colab=True),
    }

    print_header("Verification Summary")

    all_passed = True
    for name, passed in results.items():
        if passed:
            print_success(f"{name}")
        else:
            print_error(f"{name}")
            all_passed = False

    print("")
    if all_passed:
        print_success("All checks passed - Ready for FL simulation")
    else:
        print_error("Some checks failed - Fix issues before training")

main()


FLEX-Med FL Health Verification (v3.0)
Dirichlet Partitioning | Runtime Data Heterogeneity

      Step 1: Dataset Structure (v3.0 - Dirichlet Partitioning)       

  Public Anchor (20%): /content/drive/MyDrive/College/FLEX-Med/backend/datasets/cnmc/cnmc_public_anchor
    Total: 2504 images
    Leukemia (all): 1,252 (50.0%)
    Healthy  (hem): 1,252 (50.0%)
  Public Test (10%): /content/drive/MyDrive/College/FLEX-Med/backend/datasets/cnmc/cnmc_public_test
    Total: 1252 images
    Leukemia (all): 626 (50.0%)
    Healthy  (hem): 626 (50.0%)
  Local Train (70%): /content/drive/MyDrive/College/FLEX-Med/backend/datasets/cnmc/cnmc_local_train
    Total: 8774 images
    Leukemia (all): 6,615 (75.4%)
    Healthy  (hem): 2,159 (24.6%)
  
  All 3 dataset folders present (v3.0 Dirichlet partitioning ready)

           Step 2: Client Configuration (client_data.json)            

  Loaded config: /content/drive/MyDrive/College/FLEX-Med/backend/federated_learning/flex_med/utils/client_data.json
  

# **API Definition**

In [7]:
import os
import sys
import json
import logging
import subprocess
from typing import List, Tuple, Optional
import re


from fastapi import FastAPI, BackgroundTasks, HTTPException
from pydantic import BaseModel
from starlette.middleware.cors import CORSMiddleware

# ==========================================
# 1. CONFIGURATION & PATHS
# ==========================================

# Import configuration from config.py
try:
    from flex_med.utils.config import (
        
        SIMULATION_LOG_PATH,
        CLIENT_INFO_FILE_PATH,
        PYPROJECT_PATH,
        RUN_SIMULATION_SHELL_FILE_PATH,
    )
except ImportError:

    sys.path.append(os.getcwd())
    from flex_med.utils.config import (
        
        SIMULATION_LOG_PATH,
        CLIENT_INFO_FILE_PATH,
        PYPROJECT_PATH,
        RUN_SIMULATION_SHELL_FILE_PATH,
    )

os.makedirs( exist_ok=True)

# ==========================================
# 2. SERVER LOGGING SETUP
# ==========================================

logger = logging.getLogger("inference_logger")
logger.setLevel(logging.INFO)

if logger.hasHandlers():
    logger.handlers.clear()

stream_handler = logging.StreamHandler(sys.stdout)

formatter = logging.Formatter('%(asctime)s - %(levelname)s - %(message)s')
stream_handler.setFormatter(formatter)

logger.addHandler(stream_handler)

logger.info("=== Flex-Med Orchestrator Started ===")

# ==========================================
# 3. CONFIGURATION VALIDATION
# ==========================================

def validate_client_config(client) -> Tuple[bool, str]:
    """
    Validate client configuration

    Returns:
        (is_valid, error_message)
    """
    valid_models = [
        'resnet50',         # The Industry Standard
        'mobilenet_v2',     # Mobile Optimized
        'densenet121',      # High Dense Connections
        'efficientnet_b0',  # Modern Efficiency Optimizer
    ]
    if client.model_type not in valid_models:
        return False, f"Invalid model type: {client.model_type}. Must be one of {valid_models}"

    if not client.model_path:
        return False, f"Client {client.client_name} has no model_path"

    model_dir = os.path.dirname(client.model_path)
    if not os.path.exists(model_dir):
        try:
            os.makedirs(model_dir, exist_ok=True)
            logger.info(f"Created model directory: {model_dir}")
        except Exception as e:
            return False, f"Cannot create model directory: {model_dir}, error: {e}"

    return True, ""

# ==========================================
# 4. CREATE RUN SCRIPT
# ==========================================

def create_run_script(simulation_id: int, num_clients: int, supabase_url: str, supabase_key: str,
                      training_config: Optional[dict] = None):
    """
    Creates the shell script that runs the Flower simulation.

    Includes a pre-flight check that verifies num-supernodes in pyproject.toml
    matches the number of clients in client_data.json, and fixes it if needed.
    This guards against Google Drive FUSE sync delays causing stale reads.

    Args:
        simulation_id: ID of the simulation in fl_simulations table
        num_clients: Expected number of FL clients (for verification)
        supabase_url: Supabase project URL
        supabase_key: Supabase API key
        training_config: Training strategy params to export as env vars

    """
    # Build training config env var exports
    training_env_exports = ""
    if training_config:
        env_mapping = {
            "dirichlet_alpha": "FLEX_MED_DIRICHLET_ALPHA",
            "dirichlet_seed": "FLEX_MED_DIRICHLET_SEED",
            "dirichlet_min_partition_size": "FLEX_MED_DIRICHLET_MIN_PARTITION_SIZE",
            "minority_boost": "FLEX_MED_MINORITY_BOOST",
            "focal_alpha": "FLEX_MED_FOCAL_ALPHA",
            "focal_gamma": "FLEX_MED_FOCAL_GAMMA",
            "consensus_momentum": "FLEX_MED_CONSENSUS_MOMENTUM",
            "distill_weight_base": "FLEX_MED_DISTILL_WEIGHT",
            "distill_decay_rate": "FLEX_MED_DISTILL_DECAY",
            "train_loss_weight": "FLEX_MED_TRAIN_LOSS_WEIGHT",
            "distill_loss_weight": "FLEX_MED_DISTILL_LOSS_WEIGHT",
        }
        for config_key, env_var in env_mapping.items():
            if config_key in training_config:
                training_env_exports += f'export {env_var}="{training_config[config_key]}"\n'


    script_content = f"""#!/bin/bash
cd "{PROJECT_ROOT}"

echo "[Script] Starting Flex-Med Simulation at $(date)..."
echo "[Script] Simulation ID: {simulation_id}"

# Export environment variables for FL system
export FLEX_MED_SIMULATION_ID="{simulation_id}"
export SUPABASE_URL="{supabase_url}"
export SUPABASE_KEY="{supabase_key}"

# Training strategy configuration (from system_config preset)
{training_env_exports}
# Dataset paths configured via config.py defaults

# Optional: Configure Ray memory settings
export RAY_memory_usage_threshold="0.98"

echo "[Script] Training config env vars exported"
echo "[Script] FLEX_MED_DIRICHLET_ALPHA=$FLEX_MED_DIRICHLET_ALPHA"
echo "[Script] FLEX_MED_MINORITY_BOOST=$FLEX_MED_MINORITY_BOOST"
echo "[Script] FLEX_MED_CONSENSUS_MOMENTUM=$FLEX_MED_CONSENSUS_MOMENTUM"
echo "[Script] FLEX_MED_DISTILL_WEIGHT=$FLEX_MED_DISTILL_WEIGHT"

# Only install the local package in editable mode (Fast & Lightweight)
pip install -e . -q

# ==========================================
# PRE-FLIGHT: Verify num-supernodes matches client count
# Guards against Google Drive FUSE sync delays
# ==========================================
sync  # Force filesystem flush

EXPECTED_CLIENTS={num_clients}
ACTUAL_SUPERNODES=$(grep -oP 'num-supernodes\\s*=\\s*\\K\\d+' pyproject.toml | head -1)
DATA_JSON_CLIENTS=$(python3 -c "
import json
d = json.load(open('{CLIENT_INFO_FILE_PATH}'))
clients = d.get('clients', d) if isinstance(d, dict) else d
print(len(clients))
")

echo "[Pre-flight] Expected clients:       $EXPECTED_CLIENTS"
echo "[Pre-flight] pyproject num-supernodes: $ACTUAL_SUPERNODES"
echo "[Pre-flight] client_data.json clients: $DATA_JSON_CLIENTS"

if [ "$ACTUAL_SUPERNODES" != "$EXPECTED_CLIENTS" ]; then
    echo "[Pre-flight] MISMATCH detected! Fixing pyproject.toml..."
    sed -i "s/num-supernodes = [0-9]*/num-supernodes = $EXPECTED_CLIENTS/g" pyproject.toml
    sync
    NEW_VALUE=$(grep -oP 'num-supernodes\\s*=\\s*\\K\\d+' pyproject.toml | head -1)
    echo "[Pre-flight] Fixed: num-supernodes = $NEW_VALUE"
fi

if [ "$DATA_JSON_CLIENTS" != "$EXPECTED_CLIENTS" ]; then
    echo "[Pre-flight] ERROR: client_data.json has $DATA_JSON_CLIENTS clients but expected $EXPECTED_CLIENTS"
    echo "[Pre-flight] Aborting simulation."
    exit 1
fi

echo "[Pre-flight] All checks passed. Starting FL..."

# Run Flower CLI
flwr run .
"""

    with open(RUN_SIMULATION_SHELL_FILE_PATH, "w") as f:
        f.write(script_content)

    os.chmod(RUN_SIMULATION_SHELL_FILE_PATH, 0o755)
    logger.info(f"Created run_simulation.sh at {RUN_SIMULATION_SHELL_FILE_PATH}")

    return RUN_SIMULATION_SHELL_FILE_PATH

# ==========================================
# 5. BACKGROUND TASK
# ==========================================

def run_simulation_task():
    """Runs the FL simulation in background."""
    logger.info("="*70)
    logger.info("Starting Flower Simulation in Background")
    logger.info("="*70)

    try:
        logger.info("\n[Background Task] Running Flower Simulation...")

        with open(SIMULATION_LOG_PATH, "w") as log_file:
            process = subprocess.run(
                ["bash", SCRIPT_PATH],
                stdout=log_file,
                stderr=log_file,
                text=True
            )

        return_code = process.returncode

        if return_code == 0:
            logger.info("\n" + "="*70)
            logger.info("Federated Learning Simulation COMPLETED Successfully")
            logger.info("="*70)

        else:
            logger.error("\n" + "="*70)
            logger.error(f"Federated Learning Simulation FAILED (return code: {return_code})")
            logger.error("="*70)

    except Exception as e:
        logger.error(f"\nSimulation error: {e}")
        import traceback
        logger.error(traceback.format_exc())

# ==========================================
# 6. PYDANTIC MODELS (v3.0 - Removed deprecated fields)
# ==========================================

class ClientConfig(BaseModel):
    """
    Client configuration schema for v3.0

    Removed fields (deprecated):
    - has_local_data (all clients now use Dirichlet partitioning)
    - dataset_path (replaced by shared LOCAL_TRAIN_DATASET_PATH)
    """
    id: int
    created_at: str
    client_name: str
    model_type: str
    status: str
    model_path: str
    metrics: str

class StartFLRequest(BaseModel):
    """
    Request format for FL simulation with database integration and full config.

    Fields:
        simulation_id: ID of the simulation record in fl_simulations table
        clients: List of client configurations
        supabase_url: Supabase project URL for direct database access
        supabase_key: Supabase API key (anon key)
        fl_config: Flower training parameters (updates pyproject.toml)
        training_config: Training strategy parameters (exported as env vars)

    """
    simulation_id: int
    clients: List[ClientConfig]
    supabase_url: str
    supabase_key: str
    fl_config: Optional[dict] = None
    training_config: Optional[dict] = None


# ==========================================
# 7. FASTAPI APP
# ==========================================

app = FastAPI(title="Flex-Med Orchestrator")

app.add_middleware(
    CORSMiddleware,
    allow_origins=["*"],
    allow_credentials=True,
    allow_methods=["*"],
    allow_headers=["*"],
)

# ==========================================
# 8. ENDPOINTS
# ==========================================

@app.get("/")
def read_root():
    return {"message": "Flex-Med Orchestrator is running"}

def update_data_json(simulation_id: int, clients: List[ClientConfig], path: str = CLIENT_INFO_FILE_PATH):
    """
    Save client configuration to data.json with simulation metadata.

    Args:
        simulation_id: Simulation ID for tracking
        clients: List of client configurations
        path: Path to save the configuration file
    """
    try:
        client_dicts = [c.dict() for c in clients]

        # Wrap clients with simulation metadata (new format)
        config_data = {
            "simulation_id": simulation_id,
            "clients": client_dicts
        }

        os.makedirs(os.path.dirname(path), exist_ok=True)

        with open(path, "w") as f:
            json.dump(config_data, f, indent=2)

        logger.info(f"Updated data.json with simulation_id={simulation_id} and {len(clients)} clients")
        return True
    except Exception as e:
        logger.error(f"Failed to update data.json: {e}")
        return False

def update_pyproject_toml(num_clients: int, fl_config: Optional[dict] = None, path: str = PYPROJECT_PATH):
    """Update pyproject.toml with correct num-supernodes and FL parameters."""
    try:
        if not os.path.exists(path):
            logger.error(f"pyproject.toml not found at {path}")
            return False

        with open(path, 'r') as f:
            content = f.read()

        # Update num-supernodes (existing logic)
        pattern = r'(num-supernodes\s*=\s*)\d+'

        if not re.search(pattern, content):
            logger.warning("num-supernodes not found in pyproject.toml")
            return False

        old_match = re.search(r'num-supernodes\s*=\s*(\d+)', content)
        old_value = old_match.group(1) if old_match else "unknown"
        matches = re.findall(pattern, content)

        # Replace ALL occurrences (both local-simulation and app.federations sections)
        new_content = re.sub(pattern, f'\\g<1>{num_clients}', content)

        # Update num-partitions to match num-supernodes
        partitions_pattern = r'(num-partitions\s*=\s*)\d+'
        new_content = re.sub(partitions_pattern, f'\\g<1>{num_clients}', new_content)

        # Update FL config parameters in [tool.flwr.app.config] section
        if fl_config:
            param_mapping = {
                "num_server_rounds": "num-server-rounds",
                "local_epochs": "local-epochs",
                "batch_size": "batch-size",
                "lr": "lr",
                "lr_decay": "lr-decay",
                "distill_lr": "distill-lr",
                "distill_epochs": "distill-epochs",
                "temperature": "temperature",
            }
            for config_key, toml_key in param_mapping.items():
                if config_key in fl_config:
                    value = fl_config[config_key]
                    # Match both integer and float values
                    fl_pattern = rf'({re.escape(toml_key)}\s*=\s*)[\d.]+'
                    if re.search(fl_pattern, new_content):
                        new_content = re.sub(fl_pattern, f'\\g<1>{value}', new_content)
                        logger.info(f"Updated pyproject.toml: {toml_key} = {value}")
                    else:
                        logger.warning(f"Key '{toml_key}' not found in pyproject.toml")

        with open(path, 'w') as f:
            f.write(new_content)

        # Verify the write took effect by re-reading
        with open(path, 'r') as f:
            verify_content = f.read()

        verify_match = re.search(r'num-supernodes\s*=\s*(\d+)', verify_content)
        if verify_match and int(verify_match.group(1)) != num_clients:
            logger.error(f"Write verification failed! Expected {num_clients}, got {verify_match.group(1)}")
            return False

        logger.info(f"Updated pyproject.toml: num-supernodes {old_value} -> {num_clients} ({len(matches)} occurrences)")
        if fl_config:
            logger.info(f"Updated pyproject.toml FL params: {list(fl_config.keys())}")
        return True

    except Exception as e:
        logger.error(f"Failed to update pyproject.toml: {e}")
        return False

@app.post("/start_fl")
async def start_fl(
    request: StartFLRequest,
    background_tasks: BackgroundTasks,
):
    """
    Start federated learning simulation with full config from system_config.

    This endpoint now receives:
    - simulation_id: ID of the simulation record in fl_simulations table
    - clients: List of client configurations (without has_local_data/dataset_path)
    - supabase_url: Supabase URL for direct database access
    - supabase_key: Supabase API key
    - fl_config: Flower parameters (written to pyproject.toml)
    - training_config: Training strategy parameters (exported as env vars)


    All clients now use runtime Dirichlet partitioning from shared LOCAL_TRAIN_DATASET_PATH.

    Steps:
    1. Validates client configurations
    2. Saves client config to data.json (with simulation_id wrapper)
    3. Updates pyproject.toml with correct num-supernodes AND FL params
    4. Creates run script with training config env vars and pre-flight verification
    5. Triggers the FL simulation in background
    """

    simulation_id = request.simulation_id
    clients = request.clients
    num_clients = len(clients)

    logger.info(f"=" * 70)
    logger.info(f"Received FL start request")
    logger.info(f"  - Simulation ID: {simulation_id}")
    logger.info(f"  - Number of clients: {num_clients}")
    if request.fl_config:
        logger.info(f"  - FL config: {request.fl_config}")
    if request.training_config:
        logger.info(f"  - Training config: {request.training_config}")


    logger.info(f"=" * 70)

    # Log client details
    for i, client in enumerate(clients):
        logger.info(f"  Client {i}: {client.client_name:20} | {client.model_type:15}")

    # ===== VALIDATION: CHECK CLIENT CONFIGS =====
    logger.info("\n[Endpoint] Validating client configurations...")

    validation_errors = []
    for i, client in enumerate(clients):
        is_valid, error_msg = validate_client_config(client)
        if not is_valid:
            validation_errors.append(f"Client {i}: {error_msg}")

    if validation_errors:
        error_detail = "\n".join(validation_errors)
        logger.error(f"Configuration validation failed:\n{error_detail}")
        raise HTTPException(
            status_code=400,
            detail=f"Invalid client configuration:\n{error_detail}"
        )

    logger.info("All client configurations validated")

    # ===== STEP 1: UPDATE DATA.JSON (with simulation_id) =====
    logger.info("\n[Endpoint] Step 1: Updating data.json...")

    if not update_data_json(simulation_id, clients, CLIENT_INFO_FILE_PATH):
        raise HTTPException(
            status_code=500,
            detail="Failed to save client configuration to data.json"
        )

    # ===== STEP 2: UPDATE PYPROJECT.TOML (num-supernodes + FL params) =====
    logger.info("\n[Endpoint] Step 2: Updating pyproject.toml...")

    if not update_pyproject_toml(num_clients, request.fl_config, PYPROJECT_PATH):
        raise HTTPException(
            status_code=500,
            detail=f"Failed to update pyproject.toml num-supernodes to {num_clients}. "
                   f"This would cause partition ID mismatch errors during FL."
        )

    # ===== STEP 3: CREATE RUN SCRIPT (with training config env vars + pre-flight) =====
    logger.info("\n[Endpoint] Step 3: Creating run script...")
    logger.info(f"  - FLEX_MED_SIMULATION_ID={simulation_id}")
    logger.info(f"  - SUPABASE_URL={request.supabase_url}")
    logger.info(f"  - SUPABASE_KEY=***hidden***")
    if request.training_config:
        logger.info(f"  - Training config env vars: {list(request.training_config.keys())}")

    global SCRIPT_PATH
    SCRIPT_PATH = create_run_script(
        simulation_id=simulation_id,
        num_clients=num_clients,
        supabase_url=request.supabase_url,
        supabase_key=request.supabase_key,
        training_config=request.training_config,
    )

    # ===== STEP 4: SET ENVIRONMENT VARIABLES =====
    os.environ['FLEX_MED_SIMULATION_ID'] = str(simulation_id)
    os.environ['SUPABASE_URL'] = request.supabase_url
    os.environ['SUPABASE_KEY'] = request.supabase_key

    client_config_json = json.dumps({
        "simulation_id": simulation_id,
        "clients": [c.dict() for c in clients]
    })
    os.environ['FLEX_MED_CLIENT_CONFIGS'] = client_config_json

    # ===== STEP 5: TRIGGER SIMULATION =====
    logger.info("\n[Endpoint] Step 4: Starting FL simulation...")
    logger.info(f"  - Number of clients: {num_clients}")
    logger.info(f"  - All clients use Dirichlet partitioning from shared LOCAL_TRAIN_DATASET_PATH")
    logger.info(f"  - Database: Metrics will be saved to client_simulation_metrics table")
    logger.info(f"  - Pre-flight checks will verify config before flwr run")
    logger.info(f"=" * 70 + "\n")

    # Trigger the background task
    background_tasks.add_task(run_simulation_task)

    return {
        "status": "accepted",
        "message": f"Federated simulation started with {num_clients} clients",
        "simulation_id": simulation_id,
        "num_clients": num_clients,
        "config_updated": {
            "data_json": CLIENT_INFO_FILE_PATH,
            "pyproject_toml": PYPROJECT_PATH,
            "fl_config_applied": request.fl_config is not None,
            "training_config_applied": request.training_config is not None,
        },
        "log_path": SIMULATION_LOG_PATH,
        "database": {
            "metrics_table": "client_simulation_metrics",
            "simulation_table": "fl_simulations"
        },
        "version": "3.1"
    }

print("Flex-Med Orchestrator Ready (v3.1)")
print("  Endpoint: POST /start_fl")
print("  Schema: {simulation_id, clients, supabase_url, supabase_key, fl_config?, training_config?}")
print("  All clients use Dirichlet partitioning from shared LOCAL_TRAIN_DATASET_PATH")
print("  FL params written to pyproject.toml, training params exported as env vars")
print("  Pre-flight checks verify num-supernodes before each FL run")

2026-02-18 15:43:52,088 - INFO - === Flex-Med Orchestrator Started ===


INFO:inference_logger:=== Flex-Med Orchestrator Started ===


Flex-Med Orchestrator Ready (v3.1)
  Endpoint: POST /start_fl
  Schema: {simulation_id, clients, supabase_url, supabase_key, fl_config?, training_config?, dataset_paths?}
  All clients use Dirichlet partitioning from shared LOCAL_TRAIN_DATASET_PATH
  FL params written to pyproject.toml, training params exported as env vars
  Pre-flight checks verify num-supernodes before each FL run


# **NGROK Setup**

In [8]:
import nest_asyncio
import uvicorn
import subprocess
import threading
from pyngrok import ngrok
from pyngrok.process import kill_process
from fastapi import FastAPI, BackgroundTasks, HTTPException
from starlette.middleware.cors import CORSMiddleware

# 0. Cleanup
try:
    ngrok.disconnect()
    kill_process()
except:
    pass

# 1. Auth & Tunnel
# REPLACE WITH YOUR ACTUAL TOKEN
AUTH_TOKEN = "38BI51ntfzceNwdU1BTYDLnAZib_3S3SmcwQJFykk89BGBmEJ"
ngrok.set_auth_token(AUTH_TOKEN)

http_tunnel = ngrok.connect(8000)
public_url = http_tunnel.public_url

print(f"🌐 API URL: {public_url}")
print(f"🚀 POST to start FL: {public_url}/start_fl")

# 2. Run Uvicorn
nest_asyncio.apply()

def run_uvicorn():
    uvicorn.run(app, port=8000, host="0.0.0.0")

thread = threading.Thread(target=run_uvicorn)
thread.start()

🌐 API URL: https://intraspinal-agape-deidra.ngrok-free.dev
🚀 POST to start FL: https://intraspinal-agape-deidra.ngrok-free.dev/start_fl


# **View Training Logs**

In [9]:
!tail -f "/content/drive/MyDrive/College/FLEX-Med/backend/federated_learning/flex_med/utils/simulation_run.log"

[Script] Starting Flex-Med Simulation at Wed Feb 18 03:44:11 PM UTC 2026...
[Script] Simulation ID: 1
[Script] Training config env vars exported
[Script] FLEX_MED_DIRICHLET_ALPHA=2.5
[Script] FLEX_MED_MINORITY_BOOST=0.7
[Script] FLEX_MED_CONSENSUS_MOMENTUM=0.4
[Script] FLEX_MED_DISTILL_WEIGHT=0.45
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.8/7.8 MB 86.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 275.7/275.7 kB 19.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 821.0/821.0 MB 2.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.5/7.5 MB 116.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 393.1/393.1 MB 2.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.9/8.9 MB 132.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.7/23.7 MB 81.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 897.7/897.7 kB 63.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 571.0/571.0 MB 2.8 MB

INFO:inference_logger:


2026-02-18 21:37:56,480 - INFO - Federated Learning Simulation COMPLETED Successfully


INFO:inference_logger:Federated Learning Simulation COMPLETED Successfully


2026-02-18 21:37:56,482 - INFO - ======================================================================


INFO:inference_logger:======================================================================


^C


# **Kill process**

In [ ]:
# Kill any process running on port 8000
!fuser -k 8000/tcp

print("Attempted to kill processes on port 8000. Please re-run the FastAPI server cell now.")

# **Model Evaluation & Database Metrics Visualization**

In [ ]:
# ==============================================================================
# FLEX-Med — Model Evaluation & Database Metrics Visualization
# Metrics: val acc, val loss, F1, precision, recall + per-metric averages,
#          accuracy delta, data heterogeneity bar chart, confusion matrices.
# X-axis starts at Round 0 (no pre-FL data point; lines begin from Round 1).
# ==============================================================================

import os
import json
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from typing import Optional, List, Dict
from supabase import create_client

# ── Configuration ─────────────────────────────────────────────────────────────
SUPABASE_URL = os.getenv("SUPABASE_URL", "https://sxwclekerhjlyxehaywu.supabase.co")
SUPABASE_KEY = os.getenv("SUPABASE_KEY", "sb_secret_bDFZ4Q4ATX0q2TmMWibD_A_1t4j6YT2")
SIMULATION_ID: Optional[int] = None   # None → latest simulation
CLIENT_ID:     Optional[int] = None   # None → all clients

# Colors matching frontend CLIENT_COLORS
PALETTE   = ["#B80028", "#3b82f6", "#10b981", "#f59e0b", "#8b5cf6", "#ec4899", "#14b8a6", "#f97316"]
AVG_COLOR = "#1e293b"   # slate-900 for average lines

plt.rcParams.update({
    "axes.spines.top":   False,
    "axes.spines.right": False,
    "font.size":         10,
})

# ── Supabase helpers ───────────────────────────────────────────────────────────
def connect_to_supabase():
    if not SUPABASE_URL or "YOUR_SUPABASE_URL_HERE" in SUPABASE_URL:
        raise ValueError("Set SUPABASE_URL / SUPABASE_KEY env vars or fill them in above.")
    return create_client(SUPABASE_URL, SUPABASE_KEY)


def fetch_simulation_id(supabase, simulation_id: Optional[int]) -> int:
    if simulation_id is not None:
        return simulation_id
    resp = supabase.from_("fl_simulations").select("id").order("id", desc=True).limit(1).execute()
    if not resp.data:
        raise RuntimeError("No simulations found in the database.")
    return resp.data[0]["id"]


def fetch_metrics(supabase, simulation_id: int, client_id: Optional[int] = None) -> List[Dict]:
    """Fetch client_simulation_metrics joined with clients for a given simulation."""
    query = (
        supabase.from_("client_simulation_metrics")
        .select("client_id, metrics, status, clients(id, client_name, model_type)")
        .eq("simulation_id", simulation_id)
        .order("client_id")
    )
    if client_id is not None:
        query = query.eq("client_id", client_id)
    resp = query.execute()
    if not resp.data:
        raise RuntimeError(f"No metrics found for simulation {simulation_id}.")

    clients_data = []
    for row in resp.data:
        client_info = row.get("clients") or {}
        metrics = row.get("metrics") or {}
        if isinstance(metrics, str):
            metrics = json.loads(metrics)

        rounds_raw = sorted(metrics.get("rounds", []), key=lambda r: r.get("round", 0))
        round_data = []
        for r in rounds_raw:
            val = r.get("validation") or {}
            round_data.append({
                "round":     r.get("round"),
                "accuracy":  val.get("accuracy"),
                "loss":      val.get("loss"),
                "precision": val.get("precision"),
                "recall":    val.get("recall"),
                "f1_score":  val.get("f1_score"),
            })

        clients_data.append({
            "client_id":   row.get("client_id"),
            "client_name": client_info.get("client_name", f"Client {row.get('client_id')}"),
            "model_type":  client_info.get("model_type", "unknown"),
            "status":      row.get("status"),
            "rounds":      round_data,
            "post_fl":     metrics.get("global", {}).get("post_fl", {}),
            "data_het":    metrics.get("data_heterogeneity", {}),
        })
    return clients_data


# ── Series builders ────────────────────────────────────────────────────────────
def build_series(clients_data: List[Dict], metric_key: str, pct: bool = False):
    """
    Build per-client and average series with Round 0 prepended as None.
    Returns: (x_labels, per_client_series, avg_series)
      x_labels          = [0, 1, 2, ..., N]
      per_client_series = [(label, [None, v1, v2, ...]), ...]
      avg_series        = [None, avg_r1, avg_r2, ...]
    """
    all_rounds = sorted({
        r["round"] for c in clients_data for r in c["rounds"]
        if r.get(metric_key) is not None
    })
    if not all_rounds:
        return [], [], []

    x_labels = all_rounds   # [1, 2, ..., N] — axis xlim will start at 0

    per_client = []
    for c in clients_data:
        rd = {r["round"]: r.get(metric_key) for r in c["rounds"]}
        vals = [rd.get(r) for r in all_rounds]
        if pct:
            vals = [v * 100 if v is not None else None for v in vals]
        per_client.append((f"{c['client_name']} ({c['model_type']})", vals))

    avg_series = []
    for r in all_rounds:
        round_vals = [
            c_rd[r]
            for c in clients_data
            for c_rd in [{rr["round"]: rr.get(metric_key) for rr in c["rounds"]}]
            if c_rd.get(r) is not None
        ]
        if round_vals:
            avg = sum(round_vals) / len(round_vals)
            avg_series.append(avg * 100 if pct else avg)
        else:
            avg_series.append(None)

    return x_labels, per_client, avg_series


def build_delta_series(clients_data: List[Dict]):
    """
    Accuracy delta (round-to-round change %) per client + average.
    Returns: (x_labels, per_client_series, avg_series)
    Round 0 and Round 1 have None (no prior round to diff against).
    """
    all_rounds = sorted({
        r["round"] for c in clients_data for r in c["rounds"]
        if r.get("accuracy") is not None
    })
    if len(all_rounds) < 2:
        return [], [], []

    x_labels = all_rounds   # [1, 2, ..., N] — axis xlim will start at 0

    per_client = []
    for c in clients_data:
        rd = {r["round"]: r.get("accuracy") for r in c["rounds"]}
        deltas = [None]   # Round 1: no prior round to diff against
        for i in range(1, len(all_rounds)):
            prev = rd.get(all_rounds[i - 1])
            curr = rd.get(all_rounds[i])
            deltas.append((curr - prev) * 100 if (curr is not None and prev is not None) else None)
        per_client.append((f"{c['client_name']} ({c['model_type']})", deltas))

    avg_series = [None]
    for i in range(1, len(all_rounds)):
        vals = []
        for c in clients_data:
            rd = {r["round"]: r.get("accuracy") for r in c["rounds"]}
            prev = rd.get(all_rounds[i - 1])
            curr = rd.get(all_rounds[i])
            if curr is not None and prev is not None:
                vals.append((curr - prev) * 100)
        avg_series.append(sum(vals) / len(vals) if vals else None)

    return x_labels, per_client, avg_series


# ── Line drawing helper ────────────────────────────────────────────────────────
def draw_lines(ax, x_labels, per_client, avg_series, ylabel, ylim=None):
    """Draw per-client colored lines + dashed dark average line."""
    for idx, (label, vals) in enumerate(per_client):
        y = [v if v is not None else np.nan for v in vals]
        ax.plot(x_labels, y, marker="o", linewidth=1.8, markersize=4,
                color=PALETTE[idx % len(PALETTE)], label=label, alpha=0.9)

    if avg_series and any(v is not None for v in avg_series):
        y_avg = [v if v is not None else np.nan for v in avg_series]
        ax.plot(x_labels, y_avg, marker="D", linewidth=2.5, markersize=5,
                color=AVG_COLOR, linestyle="--", label="Average", zorder=5)

    ax.set_ylabel(ylabel, fontsize=9)
    ax.set_xlabel("FL Round", fontsize=9)
    if x_labels:
        ax.set_xlim(left=0)
        xtick_list = list(x_labels)
        if not xtick_list or xtick_list[0] != 0:
            xtick_list = [0] + xtick_list
        ax.set_xticks(xtick_list)
    ax.grid(True, alpha=0.06, color="black")
    if ylim is not None:
        ax.set_ylim(*ylim)
    else:
        ax.set_ylim(bottom=0)
    ax.legend(fontsize=7, loc="best", framealpha=0.85)


# ── Plot 1: 3×2 metrics grid ──────────────────────────────────────────────────
def plot_metrics_grid(clients_data, sim_id):
    fig, axes = plt.subplots(3, 2, figsize=(14, 12))
    fig.suptitle(f"FLEX-Med — Validation Metrics per Round  (Simulation {sim_id})",
                 fontsize=13, fontweight="bold", y=1.01)

    # (0,0) Val Accuracy — 0 to 100 %, anchored at (0, 0%) on the origin
    x, pc, avg = build_series(clients_data, "accuracy", pct=True)
    ax = axes[0, 0]
    ax.set_title("Validation Accuracy", fontsize=10, fontweight="bold")
    if x:
        x_acc  = [0] + list(x)
        pc_acc = [(lbl, [0.0] + list(vals)) for lbl, vals in pc]
        avg_acc = [0.0] + [v for v in avg]
        draw_lines(ax, x_acc, pc_acc, avg_acc, "Accuracy (%)", ylim=(0, 100))
    else:
        draw_lines(ax, x, pc, avg, "Accuracy (%)", ylim=(0, 100))

    # (0,1) Val Loss — from 0
    x, pc, avg = build_series(clients_data, "loss", pct=False)
    ax = axes[0, 1]
    ax.set_title("Validation Loss", fontsize=10, fontweight="bold")
    draw_lines(ax, x, pc, avg, "Loss")

    # (1,0) F1 Score — 0 to 1
    x, pc, avg = build_series(clients_data, "f1_score", pct=False)
    ax = axes[1, 0]
    ax.set_title("F1 Score", fontsize=10, fontweight="bold")
    draw_lines(ax, x, pc, avg, "F1 Score", ylim=(0, 1))

    # (1,1) Precision — 0 to 1
    x, pc, avg = build_series(clients_data, "precision", pct=False)
    ax = axes[1, 1]
    ax.set_title("Precision", fontsize=10, fontweight="bold")
    draw_lines(ax, x, pc, avg, "Precision", ylim=(0, 1))

    # (2,0) Recall — 0 to 1
    x, pc, avg = build_series(clients_data, "recall", pct=False)
    ax = axes[2, 0]
    ax.set_title("Recall", fontsize=10, fontweight="bold")
    draw_lines(ax, x, pc, avg, "Recall", ylim=(0, 1))

    # (2,1) Accuracy Delta — average only, allows negative y
    x, _, avg = build_delta_series(clients_data)
    ax = axes[2, 1]
    ax.set_title("Average Accuracy Delta (round-to-round, %)", fontsize=10, fontweight="bold")
    if x and any(v is not None for v in avg):
        y_avg = [v if v is not None else float("nan") for v in avg]
        ax.plot(x, y_avg, marker="D", linewidth=2.5, markersize=5,
                color=AVG_COLOR, linestyle="--", label="Average", zorder=5)
        ax.axhline(0, color="#64748b", linewidth=0.8, linestyle="--", alpha=0.6)
        ax.set_ylabel("\u0394 Accuracy (%)", fontsize=9)
        ax.set_xlabel("FL Round", fontsize=9)
        ax.set_xlim(left=0)
        xtick_list = list(x)
        if not xtick_list or xtick_list[0] != 0:
            xtick_list = [0] + xtick_list
        ax.set_xticks(xtick_list)
        ax.grid(True, alpha=0.06, color="black")
        ax.legend(fontsize=7, loc="best", framealpha=0.85)
    else:
        ax.text(0.5, 0.5, "Need \u2265 2 rounds", ha="center", va="center",
                transform=ax.transAxes, color="gray")

    plt.tight_layout()
    plt.show()


# ── Plot 2: Data heterogeneity bar chart ──────────────────────────────────────
def plot_data_heterogeneity(clients_data):
    valid = [c for c in clients_data if c.get("data_het") and "class_distribution" in c.get("data_het", {})]
    if not valid:
        print("[INFO] No data heterogeneity info available.")
        return
    labels   = [f"{c['client_name']}\n({c['model_type']})" for c in valid]
    leu_pcts = [c["data_het"]["class_distribution"]["leukemia_pct"] for c in valid]
    hlt_pcts = [c["data_het"]["class_distribution"]["healthy_pct"]  for c in valid]
    totals   = [c["data_het"]["total_samples"] for c in valid]
    x = np.arange(len(valid))
    fig, ax = plt.subplots(figsize=(max(6, 2 * len(valid) + 2), 4))
    ax.set_title("Data Heterogeneity per Client (Dirichlet Partitioning)",
                 fontsize=11, fontweight="bold")
    bars1 = ax.bar(x - 0.2, leu_pcts, 0.35, label="ALL (Leukemia)", color="#dc2626", alpha=0.80)
    bars2 = ax.bar(x + 0.2, hlt_pcts, 0.35, label="Healthy",        color="#22c55e", alpha=0.80)
    for bar, t in zip(bars1, leu_pcts):
        ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.8,
                f"{t:.1f}%", ha="center", va="bottom", fontsize=8)
    for bar, t in zip(bars2, hlt_pcts):
        ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.8,
                f"{t:.1f}%", ha="center", va="bottom", fontsize=8)
    for xi, tot in zip(x, totals):
        ax.text(xi, -6, f"n={tot:,}", ha="center", va="top", fontsize=8, color="#64748b")
    ax.set_xticks(x)
    ax.set_xticklabels(labels, fontsize=9)
    ax.set_ylabel("Class Distribution (%)")
    ax.set_ylim(0, 110)
    ax.legend(fontsize=9)
    ax.grid(True, axis="y", alpha=0.06, color="black")
    plt.tight_layout()
    plt.show()


# ── Plot 3: Confusion matrices (TP / FP / FN / TN) ────────────────────────────
def plot_confusion_matrices(clients_data):
    valid = [c for c in clients_data if c.get("post_fl", {}).get("confusion_matrix")]
    if not valid:
        print("[INFO] No confusion matrix data available (post-FL evaluation not yet completed).")
        return
    n = len(valid)
    fig, axes = plt.subplots(1, n, figsize=(4.5 * n, 4.5), squeeze=False)
    fig.suptitle("Post-FL Confusion Matrices (Public Test Set)",
                 fontsize=12, fontweight="bold")
    for c, ax in zip(valid, axes[0]):
        cm_raw = c["post_fl"]["confusion_matrix"]
        TP = cm_raw.get("TP", 0)
        FP = cm_raw.get("FP", 0)
        FN = cm_raw.get("FN", 0)
        TN = cm_raw.get("TN", 0)
        cm = np.array([[TP, FN], [FP, TN]])   # rows = Actual, cols = Predicted
        sns.heatmap(cm, annot=True, fmt="d", cmap="Reds", ax=ax, cbar=False,
                    xticklabels=["Pred Leukemia", "Pred Healthy"],
                    yticklabels=["Actual Leukemia", "Actual Healthy"],
                    linewidths=0.8, linecolor="white")
        acc = c["post_fl"].get("accuracy") or 0
        f1  = c["post_fl"].get("f1_score") or 0
        ax.set_title(
            f"{c['client_name']} ({c['model_type']})\nAcc = {acc:.2%}   F1 = {f1:.3f}",
            fontsize=9, fontweight="bold"
        )
    plt.tight_layout()
    plt.show()


# ── Main ──────────────────────────────────────────────────────────────────────
def main():
    supabase = connect_to_supabase()
    sim_id = fetch_simulation_id(supabase, SIMULATION_ID)
    print(f"[INFO] Simulation ID : {sim_id}")

    clients_data = fetch_metrics(supabase, sim_id, CLIENT_ID)
    print(f"[INFO] Clients       : {[c['client_name'] for c in clients_data]}")

    if not clients_data:
        print("[ERROR] No client data found.")
        return

    if any(c["rounds"] for c in clients_data):
        plot_metrics_grid(clients_data, sim_id)

    plot_data_heterogeneity(clients_data)
    plot_confusion_matrices(clients_data)


main()

In [ ]:
# ==============================================================================
# Publication-Ready FL Metrics Visualization
# Conference-paper quality. Dynamically discovers clients from DB.
# Aligned with v3.0 schema (global.post_fl, confusion_matrix as TP/FP/FN/TN).
# ==============================================================================

import os
import json
import numpy as np
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
from supabase import create_client
from typing import Optional

# ── Configuration ─────────────────────────────────────────────────────────────
SUPABASE_URL = os.getenv("SUPABASE_URL", "https://sxwclekerhjlyxehaywu.supabase.co")
SUPABASE_KEY = os.getenv("SUPABASE_KEY", "sb_secret_bDFZ4Q4ATX0q2TmMWibD_A_1t4j6YT2")
SIMULATION_ID: Optional[int] = None   # None → latest simulation
OUTPUT_DIR = "."                       # Directory to save figures

# ── Plot style ────────────────────────────────────────────────────────────────
plt.rcParams.update({
    "font.family":        "serif",
    "font.size":          10,
    "axes.labelsize":     10,
    "axes.titlesize":     11,
    "legend.fontsize":    8,
    "xtick.labelsize":    9,
    "ytick.labelsize":    9,
    "figure.dpi":         150,
    "savefig.dpi":        300,
    "axes.spines.top":    False,
    "axes.spines.right":  False,
})

# Colorblind-friendly palette (Wong 2011)
COLORS  = ["#0072B2", "#D55E00", "#009E73", "#CC79A7", "#56B4E9"]
MARKERS = ["o", "s", "^", "D", "v"]

# ── Supabase helpers ──────────────────────────────────────────────────────────
def connect():
    if not SUPABASE_URL or "YOUR_SUPABASE_URL_HERE" in SUPABASE_URL:
        raise ValueError("Set SUPABASE_URL / SUPABASE_KEY env vars.")
    return create_client(SUPABASE_URL, SUPABASE_KEY)


def fetch_data(supabase, simulation_id: Optional[int]):
    if simulation_id is None:
        r = supabase.from_("fl_simulations").select("id").order("id", desc=True).limit(1).execute()
        if not r.data:
            raise RuntimeError("No simulations found.")
        simulation_id = r.data[0]["id"]

    resp = (
        supabase.from_("client_simulation_metrics")
        .select("client_id, metrics, clients(client_name, model_type)")
        .eq("simulation_id", simulation_id)
        .order("client_id")
        .execute()
    )
    if not resp.data:
        raise RuntimeError(f"No data for simulation {simulation_id}.")

    models_data = {}
    for row in resp.data:
        client_info = row.get("clients") or {}
        name  = client_info.get("client_name", f"Client {row['client_id']}")
        model = client_info.get("model_type", "unknown")
        label = f"{name} ({model})"

        metrics = row.get("metrics") or {}
        if isinstance(metrics, str):
            metrics = json.loads(metrics)

        rounds_raw = sorted(metrics.get("rounds", []), key=lambda r: r.get("round", 0))
        acc_by_round = {}
        leu_by_round = {}
        hlt_by_round = {}
        for r in rounds_raw:
            val = r.get("validation") or {}
            rn  = r.get("round")
            if val.get("accuracy")          is not None: acc_by_round[rn] = val["accuracy"]
            if val.get("leukemia_accuracy") is not None: leu_by_round[rn] = val["leukemia_accuracy"]
            if val.get("healthy_accuracy")  is not None: hlt_by_round[rn] = val["healthy_accuracy"]

        post_fl = metrics.get("global", {}).get("post_fl", {})
        cm_raw  = post_fl.get("confusion_matrix", {})

        models_data[label] = {
            "acc_by_round": acc_by_round,
            "leu_by_round": leu_by_round,
            "hlt_by_round": hlt_by_round,
            "post_fl":      post_fl,
            "confusion_matrix": cm_raw,
        }
    return simulation_id, models_data


# ── Figure 1: Accuracy evolution ──────────────────────────────────────────────
def create_accuracy_plot(sim_id, models_data):
    fig = plt.figure(figsize=(10, 4))
    gs  = gridspec.GridSpec(1, 2, figure=fig, width_ratios=[2, 1], wspace=0.35)
    ax_main = fig.add_subplot(gs[0])
    ax_zoom = fig.add_subplot(gs[1])

    all_rounds = sorted({rn for d in models_data.values() for rn in d["acc_by_round"]})
    all_accs   = [v for d in models_data.values() for v in d["acc_by_round"].values()]
    avg_by_round = {
        rn: np.mean([d["acc_by_round"][rn] for d in models_data.values() if rn in d["acc_by_round"]])
        for rn in all_rounds
    }

    for idx, (label, data) in enumerate(models_data.items()):
        if not data["acc_by_round"]:
            continue
        rounds = sorted(data["acc_by_round"])
        accs   = [data["acc_by_round"][r] * 100 for r in rounds]
        kw = dict(color=COLORS[idx % len(COLORS)], marker=MARKERS[idx % len(MARKERS)],
                  linewidth=1.8, markersize=5, label=label)
        ax_main.plot(rounds, accs, **kw)
        ax_zoom.plot(rounds, accs, **kw)

    if len(models_data) > 1:
        avg_rounds = sorted(avg_by_round)
        avg_vals   = [avg_by_round[r] * 100 for r in avg_rounds]
        avg_kw = dict(color="black", linestyle="--", linewidth=1.2, label="Average", alpha=0.7)
        ax_main.plot(avg_rounds, avg_vals, **avg_kw)
        ax_zoom.plot(avg_rounds, avg_vals, **avg_kw)

    for ax, title in [(ax_main, f"Validation Accuracy per FL Round (Simulation {sim_id})"),
                      (ax_zoom, "Client Comparison")]:
        ax.set_xlabel("FL Round")
        ax.set_ylabel("Accuracy (%)")
        ax.set_title(title)
        ax.set_ylim(0, 100)
        ax.grid(True, alpha=0.25, linestyle=":")
        if all_rounds:
            ax.set_xticks(all_rounds)

    ax_main.legend(loc="lower right", framealpha=0.9)
    ax_zoom.legend(loc="lower right", framealpha=0.9)

    plt.suptitle("FLEX-Med Federated Learning — Heterogeneous Model Distillation",
                 fontsize=10, style="italic", y=1.01)
    plt.savefig(f"{OUTPUT_DIR}/accuracy_comparison.png", bbox_inches="tight")
    plt.savefig(f"{OUTPUT_DIR}/accuracy_comparison.pdf", bbox_inches="tight")
    print("[SAVED] accuracy_comparison.png / .pdf")
    plt.show()


# ── Figure 2: Post-FL bars + confusion matrices ───────────────────────────────
def create_post_fl_plot(sim_id, models_data):
    n = len(models_data)
    fig, axes = plt.subplots(2, n, figsize=(4 * n, 8))
    if n == 1:
        axes = axes.reshape(2, 1)
    fig.suptitle(f"Post-FL Evaluation — Simulation {sim_id}", fontsize=12, fontweight="bold")

    for idx, (label, data) in enumerate(models_data.items()):
        # ── Row 0: class-specific + overall metrics bar chart ─────────────────
        ax_bar = axes[0, idx]
        pf = data.get("post_fl", {})
        metric_vals = {
            "Accuracy":    pf.get("accuracy",          0) or 0,
            "Leukemia":    pf.get("leukemia_accuracy", 0) or 0,
            "Healthy":     pf.get("healthy_accuracy",  0) or 0,
            "F1":          pf.get("f1_score",          0) or 0,
            "Specificity": pf.get("specificity",       0) or 0,
        }
        colors_bar = ["#3498db", "#e74c3c", "#27ae60", "#9b59b6", "#f39c12"]
        bars = ax_bar.bar(list(metric_vals.keys()), list(metric_vals.values()),
                          color=colors_bar, alpha=0.85, width=0.6)
        for bar, val in zip(bars, metric_vals.values()):
            ax_bar.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.01,
                        f"{val:.3f}", ha="center", va="bottom", fontsize=7.5)
        ax_bar.set_ylim(0, 1.15)
        ax_bar.set_title(label, fontsize=8.5, fontweight="bold")
        ax_bar.set_ylabel("Score" if idx == 0 else "")
        ax_bar.grid(True, axis="y", alpha=0.25)
        ax_bar.tick_params(axis="x", rotation=15)

        # ── Row 1: confusion matrix ────────────────────────────────────────────
        ax_cm = axes[1, idx]
        cm_raw = data.get("confusion_matrix", {})
        if cm_raw:
            TP = cm_raw.get("TP", 0)
            FP = cm_raw.get("FP", 0)
            FN = cm_raw.get("FN", 0)
            TN = cm_raw.get("TN", 0)
            cm = np.array([[TP, FN], [FP, TN]])   # rows=Actual, cols=Predicted
            sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", ax=ax_cm,
                        xticklabels=["ALL", "Healthy"],
                        yticklabels=["ALL", "Healthy"],
                        cbar=False, linewidths=0.5)
            ax_cm.set_xlabel("Predicted", fontsize=9)
            ax_cm.set_ylabel("Actual" if idx == 0 else "", fontsize=9)
            ax_cm.set_title("Confusion Matrix", fontsize=8.5)
        else:
            ax_cm.text(0.5, 0.5, "No confusion\nmatrix data",
                       ha="center", va="center", transform=ax_cm.transAxes, color="gray")
            ax_cm.axis("off")

    plt.tight_layout()
    plt.savefig(f"{OUTPUT_DIR}/post_fl_metrics.png", bbox_inches="tight")
    plt.savefig(f"{OUTPUT_DIR}/post_fl_metrics.pdf", bbox_inches="tight")
    print("[SAVED] post_fl_metrics.png / .pdf")
    plt.show()


# ── Main ──────────────────────────────────────────────────────────────────────
supabase = connect()
sim_id, models_data = fetch_data(supabase, SIMULATION_ID)
print(f"[INFO] Simulation {sim_id} | Clients: {list(models_data.keys())}")

create_accuracy_plot(sim_id, models_data)
create_post_fl_plot(sim_id, models_data)

In [ ]:
# ==============================================================================
# Diagnostic — Round-by-Round Accuracy Check
# Aligned with v3.0 schema (no pre_fl; uses rounds[].validation + global.post_fl).
# Dynamically discovers clients; shows class-specific accuracy and data heterogeneity.
# ==============================================================================

import os
import json
from supabase import create_client

SUPABASE_URL = os.getenv("SUPABASE_URL", "https://sxwclekerhjlyxehaywu.supabase.co")
SUPABASE_KEY = os.getenv("SUPABASE_KEY", "sb_secret_bDFZ4Q4ATX0q2TmMWibD_A_1t4j6YT2")

supabase = create_client(SUPABASE_URL, SUPABASE_KEY)

# ── Get latest (or specified) simulation ─────────────────────────────────────
sim_resp = (
    supabase.from_("fl_simulations")
    .select("id, status, created_at")
    .order("id", desc=True)
    .limit(1)
    .execute()
)
if not sim_resp.data:
    print("[ERROR] No simulations found.")
else:
    sim    = sim_resp.data[0]
    sim_id = sim["id"]
    print(f"Simulation ID : {sim_id}")
    print(f"Status        : {sim['status']}")
    print(f"Created       : {sim['created_at']}")
    print("=" * 72)

    rows = (
        supabase.from_("client_simulation_metrics")
        .select("client_id, metrics, status, clients(client_name, model_type)")
        .eq("simulation_id", sim_id)
        .order("client_id")
        .execute()
    )

    for row in rows.data:
        ci      = row.get("clients") or {}
        name    = ci.get("client_name", f"Client {row['client_id']}")
        model   = ci.get("model_type", "unknown")
        metrics = row.get("metrics") or {}
        if isinstance(metrics, str):
            metrics = json.loads(metrics)

        print(f"\n{'─' * 72}")
        print(f"  {name} ({model}) | Status: {row['status']}")

        # ── Data heterogeneity ────────────────────────────────────────────────
        dh = metrics.get("data_heterogeneity")
        if dh and "class_distribution" in dh:
            cd = dh["class_distribution"]
            print(f"  Data  : {dh.get('total_samples')} total samples "
                  f"(train={dh.get('train_samples')}, val={dh.get('val_samples')})")
            print(f"          ALL={cd.get('leukemia_pct', '?')}%  "
                  f"Healthy={cd.get('healthy_pct', '?')}%  "
                  f"Imbalance ratio={dh.get('imbalance_ratio', '?')}:1")
        else:
            print("  Data  : [no data_heterogeneity recorded]")

        # ── Round-by-round table ──────────────────────────────────────────────
        rounds = sorted(metrics.get("rounds", []), key=lambda r: r.get("round", 0))
        if rounds:
            print(f"\n  {'Rnd':>3} | {'Val Acc':>7} | {'F1':>7} | "
                  f"{'Leu Acc':>7} | {'Hlt Acc':>7} | {'Gap':>6} | "
                  f"{'Train Loss':>10} | {'Distill Loss':>12}")
            print(f"  {'─'*3}-+-{'─'*7}-+-{'─'*7}-+-"
                  f"{'─'*7}-+-{'─'*7}-+-{'─'*6}-+-{'─'*10}-+-{'─'*12}")
            for r in rounds:
                val  = r.get("validation") or {}
                trn  = r.get("training")   or {}
                acc  = f"{val['accuracy']:.4f}"           if val.get("accuracy")          is not None else "      —"
                f1   = f"{val['f1_score']:.4f}"            if val.get("f1_score")          is not None else "      —"
                leu  = f"{val['leukemia_accuracy']:.4f}"   if val.get("leukemia_accuracy") is not None else "      —"
                hlt  = f"{val['healthy_accuracy']:.4f}"    if val.get("healthy_accuracy")  is not None else "      —"
                gap  = f"{val['class_gap']:.4f}"           if val.get("class_gap")         is not None else "     —"
                tloss= f"{trn['train_loss']:.4f}"           if trn.get("train_loss")        is not None else "         —"
                dloss= f"{trn['distill_loss']:.4f}"         if trn.get("distill_loss")      is not None else "            —"
                print(f"  {r['round']:>3} | {acc:>7} | {f1:>7} | "
                      f"{leu:>7} | {hlt:>7} | {gap:>6} | "
                      f"{tloss:>10} | {dloss:>12}")

            first  = (rounds[0].get("validation")  or {}) if rounds else {}
            final  = (rounds[-1].get("validation") or {}) if rounds else {}
            r1_acc = first.get("accuracy")
            rf_acc = final.get("accuracy")
            if r1_acc is not None and rf_acc is not None:
                delta = rf_acc - r1_acc
                sign  = "+" if delta >= 0 else ""
                print(f"\n  → Round 1 Val Accuracy : {r1_acc:.4f} ({r1_acc*100:.2f}%)")
                print(f"  → Final  Val Accuracy  : {rf_acc:.4f} ({rf_acc*100:.2f}%)")
                print(f"  → Change               : {sign}{delta*100:.2f}%")
        else:
            print("  [No round data recorded yet — FL may still be running]")

        # ── Post-FL global metrics ─────────────────────────────────────────────
        pf = metrics.get("global", {}).get("post_fl")
        if pf:
            print(f"\n  POST-FL (public test set, unseen data):")
            print(f"    Accuracy   = {pf.get('accuracy',          'N/A'):.4f}")
            print(f"    F1 Score   = {pf.get('f1_score',          'N/A'):.4f}")
            print(f"    ROC-AUC    = {pf.get('roc_auc',           'N/A'):.4f}")
            print(f"    Leukemia   = {pf.get('leukemia_accuracy', 'N/A'):.4f}  "
                  f"(Recall / Sensitivity)")
            print(f"    Healthy    = {pf.get('healthy_accuracy',  'N/A'):.4f}  "
                  f"(Specificity / TNR)")
            print(f"    Class Gap  = {pf.get('class_gap',         'N/A'):.4f}")
            cm = pf.get("confusion_matrix", {})
            if cm:
                print(f"    Confusion  : TP={cm.get('TP')}  FP={cm.get('FP')}  "
                      f"FN={cm.get('FN')}  TN={cm.get('TN')}")
        else:
            print("\n  POST-FL : [not yet recorded — FL may still be running]")

    print("\n" + "=" * 72)
    print("\nInterpretation Guide:")
    print("  Round 1 Val Acc < 60%  → Classifier-only phase (backbone frozen)")
    print("  Round 1 Val Acc > 70%  → Strong pretrained ImageNet features")
    print("  Final Acc ↑↑           → FLEX-Med distillation improving model")
    print("  Leukemia vs Healthy gap → Class imbalance effect — monitor class_gap")
    print("  Post-FL vs Final Val   → Generalisation to unseen public test set")
    print("  Distill Loss (round 1) → None is expected (no consensus yet)")